## INSTALL REQUIRED LIBRARIES

In [2]:
!pip install nltk scikit-learn pandas


##IMPORT LIBRARIES


In [3]:
import pandas as pd
import numpy as np
import nltk
import re

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [12]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## DOWNLOAD NLTK RESOURCES

In [4]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

## UPLOAD DATASET FROM Laptop

In [6]:
from google.colab import files

uploaded = files.upload()

Saving faqs - Sheet1.csv to faqs - Sheet1.csv


In [7]:
import io

# Automatically read uploaded file (CSV format expected)
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))

df.head()

,question,answer,category
0,How can I reset my password?,Go to the login page and click on 'Forgot Pass...,Account
1,I forgot my password,Use the 'Forgot Password' option on the login ...,Account
2,How do I recover my account password?,You can recover your password through the pass...,Account
3,I cannot log into my account,Please verify your email and password. If the ...,Account
4,How do I change my email address?,Navigate to Account Settings and update your e...,Account


## DATASET FORMAT REQUIREMENT

In [9]:
df = df.dropna()

# store columns safely
questions = df.iloc[:, 0]   # first column = Question
answers = df.iloc[:, 1]     # second column = Answer

# optional third column (if exists)
if df.shape[1] >= 3:
    category = df.iloc[:, 2]
    df['Category'] = category

##CLEAN TEXT FUNCTION

In [10]:
import nltk
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words]
    return " ".join(tokens)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## PREPROCESS QUESTIONS

In [13]:
df['clean_question'] = questions.apply(clean_text)

df.head()


,question,answer,category,Category,clean_question
0,How can I reset my password?,Go to the login page and click on 'Forgot Pass...,Account,Account,reset password
1,I forgot my password,Use the 'Forgot Password' option on the login ...,Account,Account,forgot password
2,How do I recover my account password?,You can recover your password through the pass...,Account,Account,recover account password
3,I cannot log into my account,Please verify your email and password. If the ...,Account,Account,log account
4,How do I change my email address?,Navigate to Account Settings and update your e...,Account,Account,change email address


## CONVERT TEXT TO VECTORS (TF-IDF)

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['clean_question'])

## CHATBOT FUNCTION (Core Logic)

In [15]:
from sklearn.metrics.pairwise import cosine_similarity

def get_response(user_query):
    user_query = clean_text(user_query)
    user_vec = vectorizer.transform([user_query])

    similarity = cosine_similarity(user_vec, X)
    idx = similarity.argmax()
    score = similarity[0][idx]

    if score < 0.2:
        return "Sorry, I couldn't find a relevant answer."

    answer = answers.iloc[idx]

    # optional category support
    if 'Category' in df.columns:
        return f"{answer}\n\n(Category: {df.iloc[idx]['Category']})"

    return answer

## RUN CHATBOT (Terminal Style)

In [16]:
print(" FAQ Chatbot is ready! Type 'exit' to stop.\n")

while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("Chatbot: Goodbye!")
        break

    response = get_response(user_input)
    print("Chatbot:", response)

 FAQ Chatbot is ready! Type 'exit' to stop.

You: Why is my account locked?
Chatbot: Accounts may be temporarily locked after multiple failed login attempts for security reasons.

(Category: Account)
You: What payment methods are accpeted
Chatbot: We accept Visa, MasterCard, PayPal, and major debit cards.

(Category: Payments)
You: What options do I have to pay for my order?
Chatbot: Yes, PayPal payments are supported during checkout.

(Category: Payments)
You: Is it safe to through website?
Chatbot: Sorry, I couldn't find a relevant answer.
You: I can't remeber my login password, what should I do
Chatbot: Go to the login page and click on 'Forgot Password'. Follow the instructions sent to your registered email address.

(Category: Account)
You: exit
Chatbot: Goodbye!


In [26]:
import gradio as gr

# -----------------------------
# YOUR BACKEND FUNCTION
# -----------------------------
def chat_ui(message, history):
    history = history or []

    response = get_response(message)

    history.append((message, response))

    return "", history


# -----------------------------
# CLEAN STABLE UI
# -----------------------------
with gr.Blocks(
    css="""
    body {
        background: #0f172a;
        margin: 0;
    }

    .gradio-container {
        max-width: 100% !important;
    }

    /* MAIN CHAT BOX */
    #chatbot {
        height: 70vh !important;
        margin: 20px;
        border-radius: 12px;
        background: white;
        overflow-y: auto !important;
    }

    /* INPUT AREA FIXED */
    .input-row {
        position: sticky;
        bottom: 0;
        display: flex;
        gap: 10px;
        padding: 15px 20px;
        background: #111827;
        z-index: 100;
    }

    #msg {
        flex: 1;
        padding: 12px;
        border-radius: 10px;
        font-size: 16px;
    }

    #send_btn {
        background: #2563eb !important;
        color: white !important;
        border-radius: 10px !important;
        font-weight: bold;
    }

    #clear_btn {
        background: #ef4444 !important;
        color: white !important;
        border-radius: 10px !important;
        margin: 10px 20px;
        font-weight: bold;
    }

    footer {
        display: none !important;
    }
    """
) as demo:

    gr.Markdown("#  FAQ Chatbot (TF-IDF)")

    chatbot = gr.Chatbot(elem_id="chatbot")

    # INPUT ROW
    with gr.Row(elem_classes="input-row"):
        msg = gr.Textbox(
            placeholder="Type your question...",
            show_label=False,
            elem_id="msg"
        )

        send_btn = gr.Button("Send", elem_id="send_btn")

    clear_btn = gr.Button("Clear Chat", elem_id="clear_btn")

    # EVENTS
    msg.submit(chat_ui, [msg, chatbot], [msg, chatbot])
    send_btn.click(chat_ui, [msg, chatbot], [msg, chatbot])
    clear_btn.click(lambda: ("", []), None, [msg, chatbot])

demo.launch()

/tmp/ipykernel_58342/4267855480.py:19: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_58342/4267855480.py:80: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(elem_id="chatbot")
/tmp/ipykernel_58342/4267855480.py:80: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(elem_id="chatbot")


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e0c83a438a9cad9f50.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
